# docagent on this machine (local)

The local twin of `colab_notebook.ipynb`. Run top to bottom; cell 1 detects the hardware and the rest follows it:

| hardware | model server | config | bge-m3 |
|---|---|---|---|
| NVIDIA GPU (Linux / WSL) | vLLM, as on Colab | `configs/colab_vllm_awq.yaml` | cuda |
| Apple Silicon Mac | Ollama (Metal) | `configs/default.yaml` | cpu |
| CPU only | Ollama (CPU, slow for drawings) | `configs/default.yaml` | cpu |

No Drive steps: the project and `store/` are already on disk and stay there. The chat is served on 127.0.0.1 only (no public share link).
Internet is needed only the first time (packages, bge-m3, the VLM weights).

## 1. Find the project and detect the hardware

In [ ]:
import os, sys, shutil, platform, subprocess
from pathlib import Path

# Project root = the nearest folder with pyproject.toml, wherever Jupyter was started from.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
os.chdir(ROOT)               # configs use paths relative to the project (store/, outputs/)
PY = sys.executable          # the kernel's Python: outside Colab, `!python` may be a different one

def has_nvidia_gpu():
    return bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0

if has_nvidia_gpu():
    HW = "cuda"              # vLLM, as on Colab
elif platform.system() == "Darwin" and platform.machine() == "arm64":
    HW = "apple"             # Ollama on Metal: vLLM does not run on Apple GPUs
else:
    HW = "cpu"               # Ollama on CPU
# HW = "cpu"                 # uncomment to force: "cuda" | "apple" | "cpu"

print(f"project:  {ROOT}\npython:   {PY}\nhardware: {HW}")
if HW == "cuda":
    !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Install the docagent package (skipped if already installed)

In [ ]:
# Checked in a fresh process so this kernel's sys.path does not hide a missing install.
installed = subprocess.run([PY, "-c", "import docagent" + (", vllm" if HW == "cuda" else "")],
                           capture_output=True).returncode == 0
# installed = False          # uncomment to reinstall

if installed:
    print("docagent already installed")
elif HW == "cuda":
    # As on Colab: vLLM brings its own torch, so install it first and the project unpinned.
    # requirements-gpu.txt: the versions of a working Colab run, recorded on Python 3.13+; older Python: unpinned
    if sys.version_info >= (3, 13):
        !"{PY}" -m pip install -q vllm==0.31.0 -e . -c requirements-gpu.txt
    else:
        !"{PY}" -m pip install -q vllm==0.31.0 && "{PY}" -m pip install -q -e .
else:
    !"{PY}" -m pip install -q -r requirements.txt && "{PY}" -m pip install -q -e .

## 3. Pick the config and start the matching model server

- **cuda:** the same vLLM start as Colab cell 6; the served name is read from the config.
- **apple / cpu:** Ollama. If it is not running, it is started in the background (log: `ollama.log`);
  the VLM is pulled once with `scripts/setup_ollama.sh` (~6 GB).

A server that is already running with the right model is reused, so re-running this cell is safe.

In [ ]:
import json, time, urllib.request, yaml

if HW == "cuda":
    CONFIG = "configs/colab_vllm_awq.yaml"   # or "configs/colab_vllm.yaml" (bf16 on GPUs with >= 20 GB)
else:
    CONFIG = "configs/default.yaml"          # Ollama, bge-m3 on cpu
# CONFIG = "configs/v2.yaml"                 # any other agent version
MAX_LEN = 8192                               # vLLM only: --max-model-len (16384 if prompts get rejected for length)
GPU_UTIL = 0.90                              # vLLM only: share of GPU memory; the rest is for bge-m3 on cuda

os.environ["DOCAGENT_CONFIG"] = CONFIG       # CLI, chat app and Agent() all read this
LLM = yaml.safe_load(open(CONFIG))["llm"]
SERVED, BASE_URL = LLM["model"], LLM["base_url"].rstrip("/")

def served_models():
    """Model names the server at BASE_URL offers, or None if nothing is listening."""
    try:
        with urllib.request.urlopen(f"{BASE_URL}/models", timeout=3) as r:
            return [m["id"] for m in json.load(r)["data"]]
    except OSError:
        return None

def is_served(models):
    return models is not None and (SERVED in models or f"{SERVED}:latest" in models)   # Ollama adds :latest

up = served_models()
print(f"{CONFIG}: backend {LLM['backend']}, model {SERVED}, server {BASE_URL}")

if LLM["backend"] == "vllm":
    if HW != "cuda":
        raise RuntimeError(f"{CONFIG} needs vLLM on an NVIDIA GPU; this machine is {HW}: use configs/default.yaml")
    if up is not None and not is_served(up):
        raise RuntimeError(f"Port 8000 already serves {up}, not {SERVED}. Stop it first: !pkill -f vllm.entrypoints")
    if is_served(up):
        print("vLLM already running")
    else:
        STARTED_VLLM = True
        if SERVED.endswith("-awq"):
            # AWQ 4-bit weights in fp16: fits a 16 GB GPU
            !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} MODEL=Qwen/Qwen2.5-VL-7B-Instruct-AWQ QUANT=awq DTYPE=half SERVED={SERVED} bash scripts/start_vllm.sh
        else:
            # start_vllm.sh picks the weights for the GPU: bf16 on >= 20 GB, AWQ below
            !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} SERVED={SERVED} bash scripts/start_vllm.sh
else:
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama is not installed. macOS: `brew install ollama` or https://ollama.com/download; "
                           "Linux: https://ollama.com/download/linux. Then re-run this cell.")
    if up is None:
        OLLAMA = subprocess.Popen(["ollama", "serve"], stdout=open(ROOT / "ollama.log", "a"),
                                  stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(60):
            time.sleep(1)
            if (up := served_models()) is not None:
                break
        else:
            raise RuntimeError("Ollama did not start in 60 s - see ollama.log")
        print("Ollama started (pid", OLLAMA.pid, ")")
    else:
        print("Ollama already running")
    if not is_served(up):
        if SERVED != "qwen2.5vl-16k":
            raise RuntimeError(f"{SERVED} is not in Ollama and scripts/setup_ollama.sh only creates qwen2.5vl-16k: "
                               f"create it with `ollama create {SERVED} -f <Modelfile>`")
        !bash scripts/setup_ollama.sh

## 4. Check the server responds

In [ ]:
models = served_models()
print(models)
assert is_served(models), f"{SERVED} is not served at {BASE_URL}"
print("ok:", SERVED)

## 5. Download the embedding model (bge-m3, once)

In [ ]:
from huggingface_hub import snapshot_download

try:
    print("bge-m3 already downloaded:", snapshot_download("BAAI/bge-m3", local_files_only=True))
except Exception:
    !"{PY}" scripts/download_models.py

## 6. Ingest the reports

Files already indexed are skipped. The first run takes a while: one VLM description per report figure (~50).
Run this **before** the chat (cell 7): the embedded vector store allows one process, and the chat app holds it.

In [ ]:
!"{PY}" -m docagent.cli ingest data/reports
!"{PY}" -m docagent.cli status

## 7. Launch the chat app (local only: http://127.0.0.1:7860)

In [ ]:
for p in (ROOT / "src", ROOT / "app"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import chat
from IPython.display import HTML, display

demo = chat.demo

demo.launch(
    server_name="127.0.0.1",   # this machine only; on a remote GPU box use `ssh -L 7860:127.0.0.1:7860 <host>`
    server_port=7860,
    share=False,               # no public gradio.live link: nothing leaves the machine
    prevent_thread_lock=True,
    inline=False,
)

url = demo.local_url
display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))

## 8. Ask a few questions in the notebook

In [ ]:
import time
from IPython.display import Markdown, display
from docagent.agent import Agent
from docagent.render import render_markdown

questions = [
    "What is the fatigue life of a 2024-T3 spar in cycles?",
    "Why did the left wing main spar fail?",
    "What caused the throttle arm to fracture?",
]
agent = chat.agent if "chat" in globals() else Agent()   # one agent per process: the vector store allows one
batch_id = "questions-" + time.strftime("%Y%m%d-%H%M%S")
for q in questions:
    resp = agent.ask(q, batch_id=batch_id)
    display(Markdown(f"### {q}\n\n" + render_markdown(resp)))
print("traces:", batch_id)

## 9. Latest requests and the trace of the last one

In [ ]:
!"{PY}" -m docagent.cli requests
!"{PY}" -m docagent.cli trace last

## 10. Spans of one request

In [ ]:
import sqlite3, pandas as pd

con = sqlite3.connect(ROOT / "store/traces.db")
REQUEST_ID = pd.read_sql("select request_id from requests order by created_at desc limit 1", con).iloc[0, 0]
# REQUEST_ID = "53b8c2c82def"   # or any id from a chat reply / `docagent requests`
print("request", REQUEST_ID)
pd.read_sql("select * from spans where request_id = ? order by started_at", con, params=(REQUEST_ID,))

## 11. Stop what this notebook started

Stops the chat app and the model server **this notebook started**; an Ollama app or a server you started yourself
is left running. `store/`, `outputs/` and `eval/results/` stay on disk: there is nothing to back up.

In [ ]:
if "chat" in globals():
    chat.demo.close()
if globals().get("STARTED_VLLM"):
    !pkill -f vllm.entrypoints
if "OLLAMA" in globals():
    OLLAMA.terminate()
print("stopped")